# INF2006 Group 12 — Run guide (Tasks 1–4 + runtime comparison)

Follow this top to bottom on a fresh EMR cluster. Every command and path below exists in this repo.
The report-ready Appendix A is a separate file: `APPENDIX_A_G12.md`.

Placeholders: `<BUCKET>` = your S3 bucket (e.g. `inf2006-g12-<studentID>`), `<ACQ>` = acquisition date `YYYY-MM-DD`,
`<RUN>` = UTC run id `YYYYMMDDTHHMMSSZ`, `<CLUSTER>` = `j-XXXXXXXX`.
Commands are bash on the EMR primary node unless marked **laptop**.

# Before you run

**What we use.** Task 1 profiles all five staged inputs. Tasks 2–4 use prices + GDELT only (SEC EDGAR and Wikinews stay in
the Task 1 corpus and its Streaming audit).

| Source | Task 1 corpus | Tasks 2–4 pipeline / model / analytics |
|---|---|---|
| Daily price CSV (`all_stocks_5yr.csv`) | Yes | **Yes** (core prediction source) |
| GDELT 1.0 events (731 zips) | Yes | **Yes** (daily features; Q2 analytics) |
| SEC EDGAR filings (311) | Yes | No |
| SEC filing inventory | Yes | No |
| Wikinews dumps (2 `.bz2`) | Yes | No |

**Where we stand.**

| Task | Status | Evidence on disk |
|---|---|---|
| 1 Corpus | Done (acquisition 2026-10-28/29; profile run 2026-11-01) | `evidence/task1/run_20261101T112713Z/`, `evidence/task1/screenshots/Task1_Screenshots.docx` |
| 2 Pipeline | Done on EMR 2026-11-09; the six comparison runs are **not yet done** | `README (Task 2).md`, `task2/cleaned/`, `task2/model_ready_v1/`, `sql/`, `evidence/task2_screenshots/` |
| 3 Model | Done 2026-11-16 in Zeppelin (`.zpln`) | `my_task3/task3final/{direction,volatility}/<run_id>/` |
| 4 Analytics | Partly done 2026-11-16 (reads S3 Task 2 outputs) | `evidence/task4/` |
| Runtime comparison | To do | written by `task2_driver/` |

**Decisions to settle before the assessed run.**

- **Storage is 100 GiB gp3 per core node, not 200.** `config/instance-groups.json` sets `SizeInGB: 100`; AWS Academy caps
  gp3 volume size and the first 200 GiB launch failed (`docs/decisions.md` D2). Quote the spec edition you were issued in
  Appendix A.
- **Spark version differs between runs.** Task 4 recorded Spark 3.5.5-amzn-1; the Task 2 handover recorded 3.5.8-amzn-0.
  Use what `spark-submit --version` prints on the run cluster (stored in each run's `run_manifest.json`).
- **Task 3 must ship as `.ipynb`.** The models currently exist as Zeppelin `.zpln`; the submission needs the notebook
  exported from EMR.
- **Appendix lettering.** Use A, B, C (AI assistance), D (contributions); p. 11 of the brief calls the contributions
  appendix "C" while p. 8 calls it "D" — note the mismatch in the report.
- **Split boundary.** The handover labels late-2017 test rows from 2018 prices; the driver drops the last five trading days
  of 2017 by default. Pick one rule, and if you use 2018 every one of the six runs must use `USE_2018_LABELS=1`.
- **Task 4 paths point at one person's bucket.** `PRICES_PATH` / `GDELT_PATH` must exist on the account used for the
  assessed run.
- **Two Task 2 SQL lines exist.** `sql/` is the handover version (pinned `LOCATION ...acq_2026-11-04`);
  `task2_driver/assets/sql/` is the parameterised line the timed runs use. Keep both, and say which produced which evidence.

# Runtime comparison: six runs

Read this before Step 3.

**Six runs: 3 dev + 3 full, all `reference`.** `CONFIG` never changes — the alternative configuration was dropped by the
teaching team, and the brief has not been updated, so say so in the report. `PROFILE` varies, and each profile runs three
times.

|  | Rep 1 | Rep 2 | Rep 3 |
|---|---|---|---|
| **Dev** — 50 stocks, 14 GDELT zips | ✓ | ✓ | ✓ |
| **Full** — 505 stocks, 731 GDELT zips | ✓ | ✓ | ✓ |

The three runs per profile give a **median and range**, which is what lets you tell a dev→full gap apart from cluster noise.
Dev vs full gives the **scaling** comparison. The **fingerprint** check shows the runs of each profile produce identical
outputs — that is the reproducibility evidence.

**What the clock covers**

- Start timing at the first pipeline stage, after the raw inputs are staged in HDFS.
- **Timed:** parsing/decompression from staged inputs, Hadoop Streaming, Hive transformations, distributed feature
  preparation, **training**, and the required output writes and checks.
- **Not timed, reported separately:** downloading, S3→HDFS copying, dependency setup, cluster start.

| Not timed | Where its time is recorded |
|---|---|
| Cluster start (~10–15 min) | `project-up.ps1` wait loop |
| Dependency setup (`pip install notebook nbconvert bash_kernel`) | manual, record it |
| S3→HDFS staging | `download_s`, `put_s`, `readback_s` in `staging_summary.txt` |
| Driver helper files, pre-flight, input lists, summaries, clean-up | driver cells marked *not timed* |

Training is inside the clock only if `TRAIN_CMD` is set — see Step 3. Leave it empty and the runs are not end-to-end.

**Files every run must leave behind** — under `~/task2_runs/<profile>_reference_rep<N>_<RUN_ID>/`:

| File | Used for |
|---|---|
| `timings.csv` | per-stage wall clock, exit code, YARN/job id |
| `run_summary.txt` / `.json` | medians, peaks, row/byte counts, PASS/FAIL |
| `reconcile.tsv` | row-count reconciliation for prices and GDELT |
| `validation.json` | dates, boundaries, duplicates, leakage columns, non-finite values |
| `features_summary.json` | split report, null counts, label balance |
| `inputs.txt` | input bytes and rows (must be the same in all three runs) |
| `resources_raw.log` | memory, YARN allocation, per-DataNode disk |
| `disk.csv` | HDFS bytes under the run folder (+ with replicas) |
| `run_manifest.json` | settings, software versions, shuffle partitions, lines per map |
| `assets_sha256.txt`, `assets_used/` | proves the same code ran in every repetition |
| `logs/<stage>.log` | per-stage output for the failures section |

`aggregate_runs.py` writes `aggregate_report.md` (per-stage median and range, resource peaks, failures, and the output
fingerprint check) and `aggregate_runs.csv` — that table **is** the runtime comparison section of the report.

**Checklist before writing that section**

- [ ] Six runs finished: 3 dev + 3 full, all `reference`
- [ ] Per-stage and end-to-end timings, with median and range **per profile**
- [ ] Output fingerprint IDENTICAL within each profile's three runs
- [ ] `inputs.txt` and `assets_sha256.txt` identical across the three runs of a profile
- [ ] Dev vs full compared on medians, and the gap judged against each profile's range
- [ ] Resource settings actually applied (`run_manifest.json`, `verify_cluster.sh` output)
- [ ] Measured memory and disk use (`resources_raw.log`, `disk.csv`, `hdfs dfsadmin -report`)
- [ ] Any failure or out-of-range run reported honestly

# Step 1 — Set up the cluster

1. **Laptop: get the repo.**

```powershell
git clone https://github.com/Hexteris/INF2006_Group12_BigData.git
cd INF2006_Group12_BigData
```

You need Git, Python 3 and the AWS CLI. `SEC_Filing_Inventory.csv` is git-ignored — copy it into the folder (there is one
at `reference/SEC_Filing_Inventory.csv`).

2. **Laptop: credentials and key.** Start Learner Lab, paste *AWS Details → AWS CLI: Show* into
   `%USERPROFILE%\.aws\credentials`, then download `vockey2.pem` into the repo root and restrict it:

```powershell
icacls vockey2.pem /inheritance:r /grant:r "${env:USERNAME}:(R)"
```

3. **Laptop: launch the reference cluster.** This creates the bucket if missing, launches with `config/instance-groups.json`
   and `config/emr-configurations.json`, waits for `WAITING`, saves `evidence/cluster_<CLUSTER>_<stamp>.json`, opens SSH
   from your IP, and tunnels JupyterHub (`https://localhost:9443`, `jovyan` / `jupyter`) and the YARN UI
   (`http://localhost:8088`).

```powershell
.\project-up.ps1 -Bucket <BUCKET> -KeyName vockey2 -PemFile vockey2.pem
```

If SSH times out later because your IP changed, reconnect with the existing cluster:

```powershell
.\project-up.ps1 -ClusterId (Get-Content .project-cluster-id) -KeyName vockey2 -PemFile vockey2.pem
```

4. **Primary node: get the code.**

```bash
sudo yum install -y git tmux
git clone https://github.com/Hexteris/INF2006_Group12_BigData.git g12 && cd g12
tmux new -s g12
export B=<BUCKET> ACQ=$(date -u +%F)
export SEC_USER_AGENT="Your Name your@email"
```

5. **Primary node: verify the configuration.** Expected: `dfs.replication=2`, 2 live DataNodes,
   `spark.dynamicAllocation.enabled=false`, 2 executors × 2 cores, 6g memory, 2g overhead, 2g driver. Output goes to
   `evidence/cluster_verify_<RUN>.txt`. Any FAIL must be fixed before the timed runs.

```bash
bash scripts/cluster/verify_cluster.sh $B
```

6. **Primary node: install the notebook kernel tooling** (needed for the timed runs; not part of the timing).

```bash
pip install --user notebook nbconvert bash_kernel
python3 -m bash_kernel.install --user
mkdir -p ~/task2_driver && cp task2_driver/INF2006_G12_task2_driver.ipynb task2_driver/run_matrix.sh ~/task2_driver/
```

Check that `hdfs`, `hive`, `hadoop`, `spark-submit` and `/usr/lib/hadoop/hadoop-streaming*.jar` exist.

# Step 2 — Build the corpus (Task 1)

**The data is already in S3** — `s3://<BUCKET>/raw/` holds every source with its manifests. Nothing is downloaded again;
each cluster copies those exact bytes into HDFS and proves they arrived unchanged.

1. **Check the S3 copy.**

```bash
aws s3 ls --recursive s3://$B/raw/ | awk '$NF !~ "/$" {n++; b += $3} END {print n " objects", b/1024^3 " GiB"}'
aws s3 ls s3://$B/raw/_manifests/
```

2. **Stage to HDFS.** Open `notebooks/INF2006_G12_stage_corpus.ipynb` in the tmux session and run cells 1–5.

| Cell | What it does |
|---|---|
| 1 Config | sets `B`, `STAGE=/mnt/g12/stage`, the 5 source rows, `STAGE_SOURCES` |
| 2 Preflight | S3 counts vs expected (1/1/311/2/731), `/mnt` free space, HDFS reachable, raw paths must not exist |
| 3 Define `stage_source` | download → local sha256 → `hdfs dfs -D dfs.replication=2 -put` → HDFS read-back sha256 → record |
| 4a–4e Stage | one cell per source; each must end with `OK:` |
| 5 Final check | per-source PASS and total file count (all 5 = 1,046 files) |
| 6 Optional | copy manifests and listing to `s3://$B/evidence/…` |

The resulting layout (identical on every teammate's cluster):

```text
/corpus/raw/prices/v1_2013-2017/acq_2026-10-28/            1 file
/corpus/raw/sec_inventory/v1/acq_2026-10-28/               1 file
/corpus/raw/sec_edgar/v1/acq_2026-10-28/                   311 files
/corpus/raw/wikinews/20261001_meta-history/acq_2026-10-29/ 2 files
/corpus/raw/gdelt/v1_events_2015-2017/acq_2026-10-29/      731 files
/corpus/raw/_manifests/<source>/<version>/acq_<date>/      staging_record.csv, staging_summary.txt
```

The `acq_` dates are the original download dates — do not re-date them. This is also the point where the runtime
comparison clock starts, so record `download_s`, `put_s` and `readback_s` from each `staging_summary.txt`.

3. **Profile the corpus.** Run the pre-step once (the `jovyan` user cannot write these paths by default), then open
   `notebooks/INF2006_G12_task1.ipynb` with the **PySpark** kernel, set `FULL_RUN = True` in cell [1] and run all cells.
   `FULL_RUN = False` is a dry run (GDELT 5 files, Wikinews first 3 M lines).

```bash
sudo -u hdfs hdfs dfs -mkdir -p /user/jovyan && sudo -u hdfs hdfs dfs -chown jovyan /user/jovyan
hdfs dfs -chmod -R 777 /corpus/raw/_manifests
```

The notebook checks: cluster config, provenance manifests, corpus inventory, provenance chain (original download → S3 →
HDFS sha256), prices (malformed rows, nulls, duplicates, coverage, invalid OHLCV, extreme moves, gaps), SEC inventory and
filings, GDELT (58 fields, event ids, SQLDATE, DATEADDED lag), Wikinews revisions/pages, cross-source summary, then copies
every JSON result to `s3://$B/evidence/task1_<RUN>/`.

Expected for the full run: prices 619,040 rows / 505 tickers; SEC 311 filings / 303 CIKs; GDELT 144,207,180 accepted lines;
Wikinews 1,954,230 revisions / 265,900 pages.

4. **Check Task 1.**

```bash
hdfs dfs -cat /corpus/raw/_manifests/*/*/acq_*/staging_summary.txt   # mismatches=0 and replication_not_2=0
hdfs dfs -du -s -h /corpus/raw                                       # ~26 GB with replicas at rep 2
```

- [ ] 5 sources staged, 1,046 files, every `staging_record.csv` row `match=yes`, replication 2
- [ ] Profiling JSON under each source's `profile/<RUN_ID>/` and in `s3://$B/evidence/task1_<RUN>/`
- [ ] `docs/corpus_inventory.md` and `docs/data_dictionary.md` updated from the run
- [ ] Notebook exported (File → Download as → Notebook) for the ZIP

# Step 3 — Run the pipeline (Task 2)

1. **Know what the driver runs.** `task2_driver/INF2006_G12_task2_driver.ipynb` (bash kernel) runs the whole pipeline
   once on a fresh Hive database and fresh HDFS folders, and writes every file the runtime comparison needs. It writes its
   own mapper/SQL/PySpark files, so the notebook reproduces the pipeline by itself.

| Stage | Engine | What it does |
|---|---|---|
| `gdelt_streaming` | Hadoop Streaming (map-only) | reads staged GDELT zips in parallel map tasks, validates structure, projects 19 columns, writes tagged gzip text |
| `hive_raw_tables` | Hive | run database + external tables over staged prices and the Streaming output |
| `hive_clean_prices` | Hive | validates prices; cleaned / rejected / out-of-scope tables |
| `hive_price_splits` | Hive | chronological splits |
| `hive_clean_gdelt` | Hive | value validation; cleaned / rejected; daily features by `DATEADDED` |
| `hive_reconcile` | Hive | row-count reconciliation, PASS/FAIL |
| `spark_features` | PySpark | rolling features, market-wide and lagged GDELT, targets, split/boundary rules |
| `spark_validate` | PySpark | dates, boundaries, duplicates, leakage columns, non-finite values, row accounting |
| `train` | any | set `TRAIN_CMD` so Task 3 training is inside the timed total |

2. **Set the parameters.** Only three matter: `PROFILE` takes both values, `CONFIG` stays `reference`, `REP` runs 1 to 3.
   The rest are driver defaults; leave them alone.

| Variable | Value / default | Meaning |
|---|---|---|
| `PROFILE` | **`dev` then `full`** | how much data; both are run |
| `CONFIG` | **`reference`** | the reference settings; never changes |
| `REP` | `1`, `2`, `3` | which repetition this is |
| `CORPUS` | `/corpus` | HDFS root |
| `RAW_PRICES_FILE` | auto | set only if several price CSVs are staged |
| `USE_2018_LABELS` | `0` | `1` labels late-2017 rows from 2018 prices |
| `KEEP_BULK` | `0` | `1` keeps the large intermediate HDFS folders |
| `TRAIN_CMD` | **required** | Task 3 training command; without it the runs are not end-to-end |
| `BUCKET` | empty | S3 target for the run evidence |

Set `TRAIN_CMD` before starting, so training is inside every timed run:

```bash
export TRAIN_CMD='spark-submit --master yarn --deploy-mode client \
  --conf spark.dynamicAllocation.enabled=false \
  --num-executors 2 --executor-cores 2 --executor-memory 6g \
  --conf spark.executor.memoryOverhead=2g --driver-memory 2g \
  ~/g12/train_task3.py --db $DB --out $OUT'
```

`~/g12/train_task3.py` must be a fixed path, because `$DB` and `$OUT` change per run.

3. **Build both input lists once.** Run driver cell 5 twice — once with `PROFILE=dev`, once with `PROFILE=full`. They are
   written to `$CORPUS/profiles/dev/` and `$CORPUS/profiles/full/`, and every run reads the same list, so the input is
   identical across that profile's three runs. Doing it now also keeps the build out of the first timed run.

4. **Run the matrix: 3 dev + 3 full.**

```bash
tmux new -s task2
cd ~/task2_driver
export BUCKET=$B            # optional: auto-upload each run's evidence
MATRIX="dev:reference full:reference" REPS="1 2 3" bash run_matrix.sh 2>&1 | tee run_6x.log
```

Do not run `run_matrix.sh` with no arguments — its default still includes the dropped `full:alt` runs.

To redo a single run:

```bash
PROFILE=dev CONFIG=reference REP=2 jupyter nbconvert --to notebook --execute INF2006_G12_task2_driver.ipynb \
  --ExecutePreprocessor.timeout=-1 --output-dir ~/task2_runs/executed_notebooks --output run_dev_reference_rep2
```

5. **Aggregate.** `run_matrix.sh` calls `aggregate_runs.py` at the end; otherwise run driver cell 9. Read two things
   closely: the **median and range** for each profile, and the **fingerprint check**, which should be `IDENTICAL` within
   each profile's three runs.

6. **Save the evidence to S3 before terminating the cluster.**

```bash
aws s3 cp --recursive ~/task2_runs s3://$B/evidence/task2_runs/ --only-show-errors
hdfs dfs -du -s -h /corpus/runs
```

7. **Check Task 2.**

| Check | Expected |
|---|---|
| `raw_prices` | 619,040 |
| `cleaned_prices` | 605,887 |
| `rejected_prices` | 23 |
| `out_of_scope_prices` | 13,130 |
| raw = cleaned + rejected + out-of-scope | 619,040 = 605,887 + 23 + 13,130 |
| GDELT | full = all 731 zips; the recorded 2026-11-09 data was a 150-date sample, so state which one a number refers to |
| Hive splits | 354,540 train / 125,316 val / 126,031 test |
| `model_ready_v1` | 349,590 / 122,801 / 126,001 (598,392 total) |

- [ ] Six runs finished with status OK
- [ ] `reconcile.tsv` all PASS in every run
- [ ] `validation.json` clean: no target or future columns in model-ready tables; GDELT enters by `DATEADDED`+1
- [ ] Fingerprint IDENTICAL within each profile's three runs
- [ ] `train` stage present in `timings.csv` for every run
- [ ] Hive tables readable from Spark (`spark.table("<db>.cleaned_prices")`)
- [ ] Job/YARN ids captured in `timings.csv` for every stage

### What the dev input contains

Dev is a real subset of the same corpus under one rule:

- **Prices:** the header plus every row of the **first 50 tickers alphabetically**.
- **GDELT:** the first 14 archives by name — `20150701` to `20150714`.

See the exact 50 tickers with the expression the driver uses:

```bash
hdfs dfs -cat /corpus/raw/prices/v1_2013-2017/acq_2026-10-28/all_stocks_5yr.csv \
  | tr -d '\r' | awk -F, 'NR>1 {print $7}' | sort -u | head -n 50
```

After a run, the list used is recorded in `dev_tickers.txt` in the run folder, `$CORPUS/profiles/dev/gdelt_paths.txt` in
HDFS, and `inputs.txt` (`prices_bytes`, `gdelt_archives`, `gdelt_zip_bytes`).

`prices_dev.csv` is built only once, so changing `DEV_TICKERS` later has no effect unless you delete
`$CORPUS/profiles/dev/prices/prices_dev.csv` first. That is what keeps dev identical across its three runs.

### Single-pass pipeline (how the 2026-11-09 handover data was produced)

This is not one of the six timed runs — the driver supersedes it — but it documents the outputs in `task2/cleaned/` and
`task2/model_ready_v1/`.

```bash
# 1. Streaming: GDELT zip to tab-separated events in HDFS
hadoop jar /usr/lib/hadoop/hadoop-streaming.jar \
  -files scripts/streaming/<mapper>.py -mapper "python3 <mapper>.py" -numReduceTasks 0 \
  -input /corpus/raw/gdelt/v1_events_2015-2017/acq_2026-10-29 \
  -output /corpus/processed/gdelt/events

# 2. Price audit (mapper + reducer)
hadoop jar /usr/lib/hadoop/hadoop-streaming.jar \
  -files scripts/streaming/price_audit_mapper.py,scripts/streaming/price_audit_reducer.py \
  -mapper "python3 price_audit_mapper.py" -reducer "python3 price_audit_reducer.py" \
  -input /corpus/raw/prices/v1_2013-2017/acq_2026-10-28 \
  -output /corpus/audit/price_audit/run_$(date -u +%Y%m%dT%H%M%SZ)

# 3. Hive, in numeric order
hive -f sql/01_raw_tables.sql
hive -f sql/02_clean_prices.sql
hive -f sql/03_price_splits.sql
hive -f sql/04_gdelt_tables.sql
hive -f sql/05_clean_gdelt.sql
hive -f sql/06_reconcile_prices.sql
```

Before any Streaming or Hive job, shut down the notebook Spark session — it holds most of the cluster's YARN memory and
other jobs wait forever. Note that `sql/01_raw_tables.sql` pins `LOCATION '/corpus/raw/prices/v1_2013-2017/acq_2026-11-04'`;
edit it to the folder you staged in Step 2 or the table will be empty. The driver's own `assets/sql/` avoids this with
`--hivevar`.

# Step 4 — Train the model (Task 3)

1. **Build the pipeline** in `notebooks/sections/task3_<topic>.py` with `# %%` cells: Imputer → VectorAssembler →
   StandardScaler → model, all fitted inside the pipeline so the preprocessing is saved with the model and learns from
   training data only.

2. **Open it in JupyterHub** (`https://localhost:9443`). First cell:

```
%%configure -f {"driverMemory": "2g"}
```

Then read `model_ready_v1`, reading train, validation and test **separately** to preserve the chronological split.

3. **Evaluate in the right order:** baselines → the small candidate set on validation → freeze the choice → score test
   once. Never tune on test.

4. **Save the model and predictions.**

```python
model.write().overwrite().save("/corpus/models/direction_v2")          # or s3a://<BUCKET>/task3/models/...
pred.write.mode("overwrite").parquet("/corpus/predictions/direction_v2")
```

5. **Export the notebook:** File → Download as → Notebook → `INF2006_G12_task3.ipynb`. The recorded runs currently live
   as Zeppelin `.zpln`; re-run the saved section in JupyterHub to get the required `.ipynb`.

6. **Reload in a fresh session and compare** (no retraining, no fitting):

```python
from pyspark.ml import PipelineModel
m = PipelineModel.load("/corpus/models/direction_v2")
new = m.transform(test).select("ticker", "trade_date", "prediction", "probability")
saved = spark.read.parquet("/corpus/predictions/direction_v2")
diff = (new.alias("a").join(saved.alias("b"), ["ticker", "trade_date"])
          .withColumn("d", abs(col("a.probability")[1] - col("b.probability")[1])))
diff.agg(max("d")).show()      # PASS if max |Δ| ≤ 1e-9
```

Record in Appendix A: checkpoint location, fresh-session date/time (UTC), tolerance (1e-9), the max |Δ| observed, and what
happens when a model file is missing or incompatible.

**Recorded runs**

| Task | Model | Selection | Final test | Evidence |
|---|---|---|---|---|
| Direction (5-session up/down) | logistic, 26 features, cutoff 0.54 | 8 candidates on 2016 validation | AUC 0.5132, balanced acc 0.5106, n=126,001 | `my_task3/task3final/direction/20261116T120657Z/` |
| Volatility (next-5-session high-volatility flag) | logistic *extra*, 14 features, threshold 0.33 | 3 candidates + majority/historical baselines | AUC 0.7455, balanced acc 0.6401, n=123,476 | `my_task3/task3final/volatility/20261116T115944Z/` |

7. **Check Task 3.**

- [ ] Target, horizon (5 trading sessions), cutoff/threshold and features documented
- [ ] Validation used for selection; test scored once after freezing
- [ ] Baseline metrics on the same observations
- [ ] `PipelineModel` saved; fresh-session reload ≤ 1e-9; dated evidence kept
- [ ] Exported `.ipynb` in the ZIP

# Step 5 — Run the analytics (Task 4)

1. **Check the paths first.** Open `evidence/task4/INF2006_G12_task4_analytics new.ipynb` (PySpark kernel) and edit cell 4 —
   the defaults still point at one person's bucket. Set your own and say in the report that these are parameters.

```python
PRICES_PATH = "s3://<BUCKET>/task2/cleaned/prices"
GDELT_PATH  = "s3://<BUCKET>/task2/cleaned/gdelt/daily_features"
OUT_DIR     = "evidence/task4"      # override with G12_OUT_DIR
WINDOW = 20; Z_HIGH, Z_LOW = 2.0, -2.0; MAX_ABS_RET = 0.5; N_GROUPS = 3
```

2. **Run it top to bottom.** Cell 1 configures the virtualenv, cell 2 installs numpy, pandas and matplotlib, then Q0
   (load + scope, asserts 605,887 price rows and 150 GDELT dates), Q1, Q2 and the run-manifest cell. The last cells
   optionally upload `OUT_DIR` to S3.

| Question | Method | Outputs |
|---|---|---|
| Q1 — unusual trading activity vs a security's own history | per-ticker `z = (log1p(volume) − mean)/sd` over the previous 20 trading rows; `|return| > 50%` set to NULL as probable split artefacts, day kept in the baseline | `q1_class_summary.csv`, `q1_zbucket_summary.csv`, `q1_monthly_flags.csv`, `q1_volume_class_and_zbucket.png`, `q1_monthly_high_share.png` |
| Q2 — do news volume/tone explain subsequent volatility | market vol = mean `|return|` per day; join GDELT on `trade_date = available_date`; terciles of `event_count` and `avg_tone`; next-day volatility compared; controlled within today's-volatility halves; within-block Spearman | `q2_news_volume_groups.csv`, `q2_tone_groups.csv`, `q2_controlled_by_today_vol.csv`, `q2_spearman_checks.csv`, `q2_block_composition.csv`, `q2_today_vs_next_day_vol.png`, `q2_next_day_vol_by_news_group.png` |

3. **Optional: add the model-reliability question** (the brief's example 3). Re-use `task3final/*/predictions`, group by
   month and by volatility decile, and report the subgroup sample size with each metric.

4. **What the report must not claim.** Association only, never causation. GDELT is global news volume, not ticker
   sentiment. 150 sampled dates are not continuous coverage, so a missing date is not zero news. Source-price-ratio and
   model diagnostics are not returns or proof of investability.

5. **Check Task 4.**

- [ ] Q1 and Q2 outputs in `evidence/task4/` with labelled charts
- [ ] `run_manifest.json` records inputs, row counts, parameters, timings
- [ ] Every conclusion tied to the query or table that supports it
- [ ] Limitations and uncertainty stated

# Step 6 — Shut down and ship

1. **Copy everything out before anything is deleted** (HDFS and the primary node's disks go with the cluster, and HDFS
   files do not restore Hive tables).

```bash
aws s3 cp --recursive ~/task2_runs   s3://$B/evidence/task2_runs/    --only-show-errors
aws s3 cp --recursive evidence        s3://$B/evidence/repo_evidence/ --only-show-errors
aws s3 cp INF2006_G12_task3.ipynb     s3://$B/evidence/notebooks/
aws s3 cp --recursive /corpus/models  s3://$B/models/                --only-show-errors
```

2. **Terminate the cluster** (laptop), then click **End Lab** in AWS Academy.

```powershell
.\project-down.ps1                     # uses .project-cluster-id
```

3. **Check the ZIP.**

- [ ] `report.pdf`, the exported `.ipynb` for Tasks 1–4, the `PipelineModel`, helper scripts and assets
- [ ] Report uses the required headings, includes all appendices, ≤ 3,000 words main text, word count on the cover
- [ ] Appendix A gives setup, data access, dependencies, configuration and execution order, including notebook filenames
- [ ] The exported notebook was re-run in a fresh EMR session and the saved model reproduces predictions without retraining
- [ ] Every verification record states check, command, expected, actual and evidence location
- [ ] Appendix A records data versions, training cutoff, software versions, random seeds and saved-model locations
- [ ] Every referenced file opens from the ZIP root; no credentials, tokens, raw datasets or machine-specific paths
- [ ] Data sources, reused code and AI assistance declared; contributions table totals 100%
- [ ] The report states all 3 repetitions, medians and ranges, and the honest failure / no-speedup outcome

# Where the evidence lives

| Report section | Cited evidence (relative path) |
|---|---|
| Corpus | `evidence/task1/run_20261101T112713Z/**`, `evidence/task1/screenshots/Task1_Screenshots.docx`, `docs/corpus_inventory.md`, `docs/data_dictionary.md` |
| Pipeline | `sql/*.sql`, `scripts/streaming/*`, `task2_driver/` (`assets_used/`), `evidence/task2_screenshots/E1–E5`, `README (Task 2).md` |
| Model | `notebooks/sections/task3_*.py`, `INF2006_G12_task3.ipynb`, `my_task3/task3final/*/{settings,metrics,predictions}` |
| Analytics | `evidence/task4/INF2006_G12_task4_analytics new.ipynb`, `evidence/task4/q1_*`, `q2_*`, `run_manifest.json` |
| Runtime comparison | `~/task2_runs/*/` → `aggregate_report.md`, `aggregate_runs.csv`, `timings.csv`, `run_manifest.json`, `resources_raw.log`, `disk.csv`, `inputs.txt` (S3 copy `s3://<BUCKET>/evidence/task2_runs/`) |
| Appendix A | `project-up.ps1`, `project-down.ps1`, `config/*.json`, `evidence/cluster_verify_*.txt`, `evidence/cluster_*.json` |

When a path, parameter or script name changes, change it here in the same commit — this notebook is the only run guide the
team maintains.